<a href="https://colab.research.google.com/github/aaminabenazir/capstone-project-part-3/blob/main/part3_modeling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [43]:
import pandas as pd
import numpy as np

# Create a dummy cleaned_data.csv file if it doesn't exist
try:
    df = pd.read_csv('cleaned_data.csv')
except FileNotFoundError:
    print("cleaned_data.csv not found. Creating a dummy file for demonstration.")
    # Create a dummy DataFrame with similar structure
    dummy_data = {
        'feature1': np.random.rand(100),
        'feature2': np.random.randint(0, 10, 100),
        'median_house_value': np.random.rand(100) * 100000 + 50000
    }
    df = pd.DataFrame(dummy_data)
    df.to_csv('cleaned_data.csv', index=False)
    df = pd.read_csv('cleaned_data.csv') # Re-read the newly created file

print(df.head())

   feature1  feature2  median_house_value
0  0.120013         1        53419.764733
1  0.529668         6        65068.752080
2  0.217784         6        99670.325817
3  0.578646         8       111010.870476
4  0.725907         6        57552.078001


In [44]:
# 1. Define feature matrix X (drop the regression target column)
# Replace 'your_target_column_name' with the actual name of your continuous target
target_col = 'median_house_value'
X = df.drop(columns=[target_col])

# 2. Define regression label
y_reg = df[target_col]

# 3. Define classification label (binarized at the median)
# This creates 0 for below median, 1 for above median
y_clf = (y_reg > y_reg.median()).astype(int)

In [45]:
# Example of One-Hot Encoding for nominal columns
# 'drop_first=True' is used to prevent multicollinearity
X = pd.get_dummies(X, drop_first=True)

# Example of Label Encoding for ordinal columns (if you have them)
# mapping = {'Low': 0, 'Medium': 1, 'High': 2}
# X['ordinal_column'] = X['ordinal_column'].map(mapping)

step 1 split the data before scalling

In [46]:
from sklearn.model_selection import train_test_split

# Split X and both labels
# We use the same random_state to ensure X_train and X_test correspond to the same rows for both labels
X_train, X_test, y_reg_train, y_reg_test = train_test_split(X, y_reg, test_size=0.2, random_state=42)
_, _, y_clf_train, y_clf_test = train_test_split(X, y_clf, test_size=0.2, random_state=42)

scale the features

In [47]:
from sklearn.preprocessing import StandardScaler

# Initialize the scaler
scaler = StandardScaler()

# Fit only on the training set
# This calculates the mean and standard deviation of the training data
scaler.fit(X_train)

# Transform both sets using the statistics from the training set
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

step 1 linear regression(baseline)
* we will build and compare our regression models. We will use Linear Regression as our baseline and Ridge Regression to introduce L2 regularization, which helps prevent overfitting by penalizing large coefficients.

In [48]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# Initialize and train the model
lr = LinearRegression()
lr.fit(X_train_scaled, y_reg_train)

# Predict on the test set
y_pred_lr = lr.predict(X_test_scaled)

# Calculate metrics
mse_lr = mean_squared_error(y_reg_test, y_pred_lr)
r2_lr = r2_score(y_reg_test, y_pred_lr)

print(f"Linear Regression - MSE: {mse_lr:.4f}, R2: {r2_lr:.4f}")

Linear Regression - MSE: 860362222.7672, R2: -0.0486


step 2

In [49]:
from sklearn.linear_model import Ridge

# Initialize and train with alpha=1.0
ridge = Ridge(alpha=1.0)
ridge.fit(X_train_scaled, y_reg_train)

# Predict
y_pred_ridge = ridge.predict(X_test_scaled)

# Calculate metrics
mse_ridge = mean_squared_error(y_reg_test, y_pred_ridge)
r2_ridge = r2_score(y_reg_test, y_pred_ridge)

print(f"Ridge Regression - MSE: {mse_ridge:.4f}, R2: {r2_ridge:.4f}")

Ridge Regression - MSE: 859808852.2661, R2: -0.0480


step3 featurebimportance anaysis
*we need to identify the top 3 features with the largest absolute coefficient values from your Linear Regression model.

In [50]:
# Create a series of coefficients
coef_df = pd.DataFrame({'Feature': X.columns, 'Coefficient': lr.coef_})
coef_df['Abs_Coef'] = coef_df['Coefficient'].abs()

# Sort and print top 3
top_3 = coef_df.sort_values(by='Abs_Coef', ascending=False).head(3)
print(top_3)

    Feature  Coefficient     Abs_Coef
0  feature1 -2088.754274  2088.754274
1  feature2  1833.518336  1833.518336


phase 4 classification (logistic regression)
*step 1 ->First, check your counts. If one class is below 35%, use ⁠class_weight='balanced'⁠.

*

In [51]:
# Check imbalance
print(y_clf_train.value_counts(normalize=True))

# Train Logistic Regression with balanced class weights
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, roc_auc_score
import matplotlib.pyplot as plt

# Using class_weight='balanced' automatically adjusts weights inversely proportional to class frequencies
clf = LogisticRegression(max_iter=1000, class_weight='balanced')
clf.fit(X_train_scaled, y_clf_train)

median_house_value
0    0.5
1    0.5
Name: proportion, dtype: float64


LogisticRegression(class_weight='balanced', max_iter=1000)

step 2 evaluation

step 3 decision threshold sensitivity

In [52]:
from sklearn.metrics import precision_score, recall_score, f1_score

# Get probabilities from the base classifier
y_prob_clf = clf.predict_proba(X_test_scaled)[:, 1]

thresholds = [0.3, 0.4, 0.5, 0.6, 0.7]
results = []

for t in thresholds:
    # Create predictions based on the custom threshold
    y_pred_t = (y_prob_clf >= t).astype(int)
    results.append({
        'Threshold': t,
        'Precision': precision_score(y_clf_test, y_pred_t),
        'Recall': recall_score(y_clf_test, y_pred_t),
        'F1': f1_score(y_clf_test, y_pred_t)
    })

results_df = pd.DataFrame(results)
print(results_df)

   Threshold  Precision  Recall        F1
0        0.3   0.500000     1.0  0.666667
1        0.4   0.473684     0.9  0.620690
2        0.5   0.400000     0.4  0.400000
3        0.6   0.000000     0.0  0.000000
4        0.7   0.000000     0.0  0.000000


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Phase 5 involves two critical analytical steps: testing the impact of the regularization parameter (⁠C⁠) and calculating a bootstrap confidence interval to determine if the performance difference between models is statistically reliable.

step 1 regularization experiment
*we compare the base line c=1.0 model against a strongly regularized c=0.01 model

In [53]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

# Train second model with strong regularization (C=0.01)
clf_reg = LogisticRegression(max_iter=1000, C=0.01, class_weight='balanced')
clf_reg.fit(X_train_scaled, y_clf_train)

# Predict probabilities for the new model
y_prob_reg = clf_reg.predict_proba(X_test_scaled)[:, 1]

# Compare AUCs
auc_base = roc_auc_score(y_clf_test, y_prob_clf)
auc_reg = roc_auc_score(y_clf_test, y_prob_reg)

print(f"Baseline (C=1.0) AUC: {auc_base:.4f}")
print(f"Regularized (C=0.01) AUC: {auc_reg:.4f}")

Baseline (C=1.0) AUC: 0.3800
Regularized (C=0.01) AUC: 0.3800


step 2 booststrap confidence interval
* this deternmine or just due to change .

In [54]:
n_iterations = 500
auc_diffs = []

# Get predictions for both models
y_prob_base = clf.predict_proba(X_test_scaled)[:, 1]
y_prob_reg = clf_reg.predict_proba(X_test_scaled)[:, 1]

# Convert arrays to pandas for easier indexing
y_test_arr = y_clf_test.values

for i in range(n_iterations):
    # Sample indices with replacement
    indices = np.random.choice(len(y_test_arr), size=len(y_test_arr), replace=True)

    # Calculate AUC for this bootstrap sample
    auc_base_sample = roc_auc_score(y_test_arr[indices], y_prob_base[indices])
    auc_reg_sample = roc_auc_score(y_test_arr[indices], y_prob_reg[indices])

    auc_diffs.append(auc_base_sample - auc_reg_sample)

# Calculate 2.5th and 97.5th percentiles
lower = np.percentile(auc_diffs, 2.5)
upper = np.percentile(auc_diffs, 97.5)

print(f"95% Confidence Interval for AUC difference: ({lower:.4f}, {upper:.4f})")

95% Confidence Interval for AUC difference: (0.0000, 0.0000)


part 3


In [55]:
from sklearn.tree import DecisionTreeClassifier
# This trains the model using your data from Part 2
clf = DecisionTreeClassifier(max_depth=None)
clf.fit(X_train_scaled, y_clf_train)

DecisionTreeClassifier()

In [56]:
import joblib
# Use the name of the variable you just trained (e.g., clf or best_pipeline)
joblib.dump(clf, 'best_model.pkl')

['best_model.pkl']

In [57]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# 1. Baseline Model (No constraints - prone to overfitting)
# This model tries to learn every single detail of your training data.
clf_baseline = DecisionTreeClassifier(max_depth=None)
clf_baseline.fit(X_train_scaled, y_clf_train)

# 2. Controlled Model (With constraints)
# By limiting the depth, we force the tree to be simpler and generalize better.
clf_controlled = DecisionTreeClassifier(max_depth=5, min_samples_split=20)
clf_controlled.fit(X_train_scaled, y_clf_train)

# Checking accuracy
print("Baseline Train Accuracy:", accuracy_score(y_clf_train, clf_baseline.predict(X_train_scaled)))
print("Controlled Train Accuracy:", accuracy_score(y_clf_train, clf_controlled.predict(X_train_scaled)))

Baseline Train Accuracy: 1.0
Controlled Train Accuracy: 0.625


In [58]:
# Train two trees to compare the formulas
clf_gini = DecisionTreeClassifier(max_depth=5, criterion='gini')
clf_entropy = DecisionTreeClassifier(max_depth=5, criterion='entropy')

clf_gini.fit(X_train_scaled, y_clf_train)
clf_entropy.fit(X_train_scaled, y_clf_train)

print("Gini Accuracy:", clf_gini.score(X_test_scaled, y_clf_test))
print("Entropy Accuracy:", clf_entropy.score(X_test_scaled, y_clf_test))

Gini Accuracy: 0.45
Entropy Accuracy: 0.4


In [59]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

# n_estimators=100 means it will build 100 trees
rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
rf.fit(X_train_scaled, y_clf_train)

# Check performance
y_pred = rf.predict_proba(X_test_scaled)[:, 1]
print("Random Forest ROC-AUC:", roc_auc_score(y_clf_test, y_pred))

# Get top 5 important features
importances = rf.feature_importances_
# (You will use this to identify the features to remove in Task 4b)

Random Forest ROC-AUC: 0.71


In [60]:
import joblib

# Replace 'rf' with whatever model you decide is your "best_model"
joblib.dump(rf, 'best_model.pkl')

# How to load and use it later:
loaded_model = joblib.load('best_model.pkl')
# Make a prediction with 2 sample rows
# sample_data = X_test_scaled[:2]
# print(loaded_model.predict(sample_data))